# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q statsmodels
    print('Packages installed.')


# Group Comparisons: T-tests & ANOVA

A fundamental task in social science is comparing means between groups (e.g., Treatment vs Control). This notebook goes beyond simple p-values to include:

1.  **Assumption Checking**: Normality and Homogeneity of Variance.
2.  **Effect Size**: Cohen's d (t-test) and Eta-squared (ANOVA).
3.  **Visualization**: Boxplots with jitter points to show raw data.
4.  **Post-hoc Testing**: Determining exactly *which* groups differ.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sns.set_style('whitegrid')

## 1. Independent Samples T-Test
Scenario: Comparing test scores between two teaching methods (Method A vs Method B).

In [ ]:
np.random.seed(42)

# Generate data
# Method A: Mean 75, SD 10
group_a = np.random.normal(75, 10, 40)
# Method B: Mean 82, SD 10
group_b = np.random.normal(82, 10, 40)

df_ttest = pd.DataFrame({
    'Score': np.concatenate([group_a, group_b]),
    'Method': ['Method A'] * 40 + ['Method B'] * 40
})

plt.figure(figsize=(8, 5))
sns.boxplot(x='Method', y='Score', data=df_ttest, palette='pastel')
sns.stripplot(x='Method', y='Score', data=df_ttest, color='black', alpha=0.5, jitter=True)
plt.title('Test Scores by Method')
plt.show()

### Assumption Checks
1. **Normality**: Shapiro-Wilk test (p > 0.05 indicates normal distribution).
2. **Homogeneity of Variance**: Levene's test (p > 0.05 indicates equal variances).

In [ ]:
# Normality
stat_a, p_a = stats.shapiro(group_a)
stat_b, p_b = stats.shapiro(group_b)
print(f"Shapiro Method A: p={p_a:.3f}")
print(f"Shapiro Method B: p={p_b:.3f}")

# Levene
stat_lev, p_lev = stats.levene(group_a, group_b)
print(f"Levene Test: p={p_lev:.3f}")

### Run T-Test

In [ ]:
t_stat, p_val = stats.ttest_ind(group_a, group_b)
print(f"T-Statistic: {t_stat:.3f}, P-Value: {p_val:.4f}")

# Effect Size (Cohen's d)
d = (np.mean(group_a) - np.mean(group_b)) / np.sqrt((np.std(group_a, ddof=1)**2 + np.std(group_b, ddof=1)**2) / 2)
print(f"Cohen's d: {d:.3f}")

## 2. One-Way ANOVA
Scenario: Comparing stress levels across three occupation types (Tech, Health, Education).

In [ ]:
# Generate data
tech = np.random.normal(60, 15, 50)
health = np.random.normal(70, 15, 50)
edu = np.random.normal(65, 15, 50)

df_anova = pd.DataFrame({
    'Stress': np.concatenate([tech, health, edu]),
    'Occupation': ['Tech'] * 50 + ['Health'] * 50 + ['Education'] * 50
})

plt.figure(figsize=(9, 6))
sns.violinplot(x='Occupation', y='Stress', data=df_anova, palette='muted')
plt.title('Stress Levels by Occupation')
plt.show()

### Run ANOVA

In [ ]:
model = ols('Stress ~ C(Occupation)', data=df_anova).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

# Calculate Eta Squared (Effect Size)
es = anova_table['sum_sq'].iloc[0] / (anova_table['sum_sq'].iloc[0] + anova_table['sum_sq'].iloc[1])
print(f"\nEta Squared: {es:.3f}")

### Post-hoc Analysis (Tukey's HSD)
If the ANOVA is significant, we use Tukey's Honestly Significant Difference test to see which specific pairs differ.

In [ ]:
tukey = pairwise_tukeyhsd(endog=df_anova['Stress'],
                          groups=df_anova['Occupation'],
                          alpha=0.05)

print(tukey)

## Conclusion
Reporting results should include the test statistic, degrees of freedom, p-value, and effect size. For example: "There was a significant effect of occupation on stress levels, F(2, 147) = [F-val], p < .05, eta^2 = [ES]."